<a href="https://colab.research.google.com/github/SebaAcunaC/YOLO11-Vigilancia-Urbana-USACH/blob/main/Semana-5-Optimizacion/Semana5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# PROYECTO YOLO11 - SEMANA 5: OPTIMIZACIÓN Y COMPARATIVA MULTI-RESOLUCIÓN
# Objetivo: Encontrar el "Sweet Spot" de resolución para superar los 30 FPS.
# ==============================================================================

# 1. SETUP E INSTALACIÓN
!pip install ultralytics pandas matplotlib -q

import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 2. CARGA DE MODELOS (Duelo Técnico)
# Descargamos tus pesos entrenados desde el enlace público de Drive
!wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=TU_ID_DE_DRIVE' -O best.pt

model_v11 = YOLO('best.pt')  # Tu modelo optimizado
model_v8 = YOLO('yolov8n.pt') # Modelo base para comparación

# 3. CONFIGURACIÓN DE PRUEBAS
# Usamos Stride 32 para optimizar el uso de los núcleos CUDA de la GPU
resoluciones = [1088, 640, 320]
input_folder = 'videos_prueba'
output_base = 'resultados_optimizados'
os.makedirs(output_base, exist_ok=True)

print("Entorno listo. Sube tus videos .mp4 a la carpeta 'videos_prueba'.")

# 4. BUCLE DE BENCHMARKING AUTOMATIZADO
resultados_data = []

if os.path.exists(input_folder) and any(f.endswith('.mp4') for f in os.listdir(input_folder)):
    videos = [f for f in os.listdir(input_folder) if f.endswith('.mp4')]

    for res in resoluciones:
        print(f"\n--- Probando Resolución: {res}p ---")
        for video_name in videos:
            video_path = os.path.join(input_folder, video_name)
            video_id = video_name.split('.')[0]

            # Inferencia YOLO11 con modo Stream para proteger RAM
            res_11 = model_v11.predict(source=video_path, imgsz=res, stream=True, save=True,
                                      project=f"{output_base}/{video_id}", name=f"YOLO11_{res}p", conf=0.5)
            for r in res_11: last_r11 = r
            t11 = sum(last_r11.speed.values())
            resultados_data.append({'Modelo': 'YOLO11', 'Resolucion': res, 'FPS': 1000/t11})

            # Inferencia YOLOv8
            res_8 = model_v8.predict(source=video_path, imgsz=res, stream=True, save=True,
                                    project=f"{output_base}/{video_id}", name=f"YOLOv8_{res}p", conf=0.5)
            for r in res_8: last_r8 = r
            t8 = sum(last_r8.speed.values())
            resultados_data.append({'Modelo': 'YOLOv8', 'Resolucion': res, 'FPS': 1000/t8})

    # 5. GENERACIÓN DE LA GRÁFICA DE INGENIERÍA
    df = pd.DataFrame(resultados_data)
    df_avg = df.groupby(['Modelo', 'Resolucion'])['FPS'].mean().unstack(level=0)

    plt.figure(figsize=(10, 6))
    plt.plot(df_avg.index, df_avg['YOLO11'], marker='o', label='YOLO11 (USACH)', linewidth=2)
    plt.plot(df_avg.index, df_avg['YOLOv8'], marker='s', label='YOLOv8 (Base)', linestyle='--')
    plt.axhline(y=30, color='r', linestyle=':', label='Meta Tiempo Real (30 FPS)')
    plt.title('Trade-off de Ingeniería: Resolución vs. FPS Reales')
    plt.xlabel('Resolución (Píxeles)')
    plt.ylabel('Frames Per Second (FPS)')
    plt.gca().invert_xaxis()
    plt.grid(True, alpha=0.5)
    plt.legend()
    plt.savefig('grafica_optimizacion_s5.png', dpi=300)
    plt.show()